# Module 5: LULC Changes Around Major Water Bodies

**Task 5**: Analyze LULC changes around major water bodies using multi-ring buffers 2, 4, 8, 10 km (State level).

**Deliverables**:
- Buffer zone shapefiles
- Ring-wise LULC composition tables
- Stacked area/bar charts per ring per year
- Buffer overlay maps
- Encroachment pattern interpretation

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterstats import zonal_stats

from config import (
    STATES, YEARS, LULC_CLASSES, LULC_COLORS, NUM_CLASSES,
    PROCESSED_DIR, FIGURES_DIR, LULC_DIR, BOUNDARIES_DIR, GEE_EXPORTS_DIR,
    WATER_BUFFER_DISTANCES_M, MAJOR_WATER_BODY_THRESHOLD_HA,
    PIXEL_AREA_KM2, TIER2_SCALE,
)
from scripts.buffer_analysis import create_water_body_buffers
from scripts.visualization import save_figure, plot_buffer_lulc_stacked
from scripts.lulc_utils import save_composition_csv

print('Imports successful!')

## 5.1: Select Major Water Bodies

In [ ]:
major_water_bodies = {}

for state_key, state_cfg in STATES.items():
    threshold = MAJOR_WATER_BODY_THRESHOLD_HA[state_key]
    
    # Use 2016 baseline
    wb_path = PROCESSED_DIR / 'water_bodies' / f'{state_key}_water_classified_2016.geojson'
    if not wb_path.exists():
        wb_path = GEE_EXPORTS_DIR / f'{state_key}_water_bodies_2016.geojson'
    
    if not wb_path.exists():
        print(f'⚠️ Water bodies not found for {state_cfg["name"]} — run Module 4 first')
        continue
    
    gdf = gpd.read_file(wb_path)
    
    # Compute area if needed
    if 'area_ha' not in gdf.columns:
        gdf_utm = gdf.to_crs(epsg=state_cfg['utm_epsg'])
        gdf['area_ha'] = gdf_utm.geometry.area / 10000
    
    major = gdf[gdf['area_ha'] >= threshold].copy()
    major_water_bodies[state_key] = major
    
    print(f'{state_cfg["name"]}: {len(major)} major water bodies (>{threshold} ha)')
    if len(major) > 0:
        print(f'  Largest: {major["area_ha"].max():.1f} ha')
        print(f'  Total area: {major["area_ha"].sum():.1f} ha')

## 5.2: Create Multi-Ring Buffers

In [ ]:
buffer_rings = {}

for state_key, state_cfg in STATES.items():
    if state_key not in major_water_bodies or len(major_water_bodies[state_key]) == 0:
        print(f'⚠️ No major water bodies for {state_cfg["name"]}')
        continue
    
    print(f'Creating buffer rings for {state_cfg["name"]}...')
    ring_gdf = create_water_body_buffers(
        major_water_bodies[state_key],
        crs_utm=state_cfg['utm_epsg'],
        distances_m=WATER_BUFFER_DISTANCES_M
    )
    
    buffer_rings[state_key] = ring_gdf
    
    # Save buffer shapefiles
    outdir = PROCESSED_DIR / 'buffers'
    outdir.mkdir(parents=True, exist_ok=True)
    ring_gdf.to_file(outdir / f'{state_key}_water_buffers.geojson', driver='GeoJSON')
    
    print(f'  ✅ {len(ring_gdf)} buffer rings created')
    display(ring_gdf[['ring_label', 'distance_inner_m', 'distance_outer_m']])

## 5.3: Zonal LULC Composition per Ring

In [ ]:
ring_compositions = {}

for state_key, state_cfg in STATES.items():
    if state_key not in buffer_rings:
        continue
    
    rings = buffer_rings[state_key]
    all_rows = []
    
    for year in YEARS:
        raster_path = LULC_DIR / f'{state_key}_30m_{year}.tif'
        if not raster_path.exists():
            print(f'⚠️ Raster not found: {raster_path.name}')
            continue
        
        print(f'  Computing ring stats: {state_cfg["name"]} {year}...')
        
        # Zonal stats — categorical count per ring
        stats = zonal_stats(
            rings, str(raster_path),
            categorical=True, nodata=255
        )
        
        for i, (_, ring_row) in enumerate(rings.iterrows()):
            row = {
                'year': year,
                'ring_label': ring_row['ring_label'],
            }
            ring_stats = stats[i] if i < len(stats) else {}
            total = sum(ring_stats.values()) if ring_stats else 1
            
            for cls_val in range(NUM_CLASSES):
                count = ring_stats.get(cls_val, 0)
                row[LULC_CLASSES[cls_val]] = (count / total * 100) if total > 0 else 0
            
            all_rows.append(row)
    
    if all_rows:
        comp_df = pd.DataFrame(all_rows)
        ring_compositions[state_key] = comp_df
        
        save_composition_csv(
            comp_df,
            PROCESSED_DIR / 'buffers' / f'{state_key}_water_buffer_composition.csv'
        )
        display(comp_df)

## 5.4: Visualization

In [ ]:
# Stacked bar charts per ring per year
for state_key, state_cfg in STATES.items():
    if state_key not in ring_compositions:
        continue
    plot_buffer_lulc_stacked(
        ring_compositions[state_key], state_cfg['name'], 'Water Body'
    )

print('✅ Buffer charts generated.')

In [ ]:
# Map: buffer rings overlaid on LULC
for state_key, state_cfg in STATES.items():
    if state_key not in buffer_rings or state_key not in major_water_bodies:
        continue
    
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # Plot boundary
    boundary_file = BOUNDARIES_DIR / f'{state_key}_district_boundaries.geojson'
    if boundary_file.exists():
        gpd.read_file(boundary_file).plot(ax=ax, color='#f5f5f5', edgecolor='grey', linewidth=0.3)
    
    # Plot buffer rings
    ring_colors = ['#fee5d9', '#fcae91', '#fb6a4a', '#cb181d']
    rings = buffer_rings[state_key]
    for i, (_, ring) in enumerate(rings.iterrows()):
        color = ring_colors[i % len(ring_colors)]
        gpd.GeoDataFrame([ring], crs=rings.crs).plot(
            ax=ax, color=color, alpha=0.4, edgecolor=color, linewidth=0.5
        )
    
    # Plot water bodies
    major_water_bodies[state_key].plot(ax=ax, color='#2171b5', alpha=0.9)
    
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#2171b5', label='Major Water Body')]
    for i, (_, ring) in enumerate(rings.iterrows()):
        legend_elements.append(Patch(facecolor=ring_colors[i % len(ring_colors)],
                                     alpha=0.4, label=ring['ring_label']))
    ax.legend(handles=legend_elements, loc='lower right')
    
    ax.set_title(f'Water Body Buffer Zones — {state_cfg["name"]}', fontsize=14)
    ax.set_axis_off()
    save_figure(fig, f'{state_key}_water_buffer_map.png', 'water_analysis')
    plt.show()

print('\n✅ Module 5 complete.')

---
## Summary
- ✅ Major water bodies selected (>100ha Maharashtra, >50ha Sikkim)
- ✅ Multi-ring buffers at 2, 4, 8, 10 km created
- ✅ LULC composition per ring per year computed
- ✅ Stacked bar charts showing land-use change near water bodies
- ✅ Buffer overlay maps

**Next**: `06_urban_expansion.ipynb`